In [1]:
%matplotlib inline

In [2]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt

# Flight Delay EDA

This notebook explores the 2024 US domestic flight dataset (~7.1M flights, 35 columns) as the foundation for an **inflight arrival-delay prediction system**. Before any modeling we clean and understand the raw data. In this exploratory pass we load the data, quantify missing values, and remove records that cannot carry a delay label, so that later steps work from a clean, well-understood table. The aim is to understand data quality, the distribution of delays, and which fields are usable as predictive features.

## Data cleaning

### Step one: import the data

We load the full 2024 flight file into a pandas DataFrame. `low_memory=False` tells pandas to scan the whole file before inferring column types, which avoids mixed-type warnings on the larger text columns. The resulting table has **7,079,081 rows and 35 columns**, which we confirm with `.shape`.

In [3]:
flights = pd.read_csv("../data/raw//flight_data_2024.csv",low_memory=False)

In [4]:
flights.shape

(7079081, 35)

In [5]:
flights.dtypes

year                     int64
month                    int64
day_of_month             int64
day_of_week              int64
fl_date                 object
op_unique_carrier       object
op_carrier_fl_num      float64
origin                  object
origin_city_name        object
origin_state_nm         object
dest                    object
dest_city_name          object
dest_state_nm           object
crs_dep_time             int64
dep_time               float64
dep_delay              float64
taxi_out               float64
wheels_off             float64
wheels_on              float64
taxi_in                float64
crs_arr_time             int64
arr_time               float64
arr_delay              float64
cancelled                int64
cancellation_code       object
diverted                 int64
crs_elapsed_time       float64
actual_elapsed_time    float64
air_time               float64
distance               float64
carrier_delay            int64
weather_delay            int64
nas_dela

### Step two: null check

We count missing values per column, as both a raw count and a percentage of all rows, then sort by null percentage to see which fields are incomplete. Most columns are fully populated. The missing values are concentrated in flight-timing fields (`dep_time`, `arr_time`, `arr_delay`, `taxi_out`, etc.) and in `cancellation_code`. These nulls are **not random** — they correspond almost entirely to cancelled flights, which never departed or arrived, and to `cancellation_code` being populated only for those cancelled rows.

In [6]:

null_counts = flights.isna().sum()
null_pct = (null_counts / len(flights) * 100).round(2)

null_summary = (
    pd.DataFrame({"null_count": null_counts, "null_pct": null_pct})
    .sort_values("null_pct", ascending=False)
)
null_summary

,null_count,null_pct
cancellation_code,6982766,98.64
actual_elapsed_time,113814,1.61
arr_delay,113814,1.61
air_time,113814,1.61
taxi_in,97856,1.38
wheels_on,97856,1.38
arr_time,97854,1.38
taxi_out,95734,1.35
wheels_off,95734,1.35
dep_time,92659,1.31


In [7]:
null_summary[null_summary["null_count"] > 0]

,null_count,null_pct
cancellation_code,6982766,98.64
actual_elapsed_time,113814,1.61
arr_delay,113814,1.61
air_time,113814,1.61
taxi_in,97856,1.38
wheels_on,97856,1.38
arr_time,97854,1.38
taxi_out,95734,1.35
wheels_off,95734,1.35
dep_time,92659,1.31


### Step three: where do the nulls come from?

The null counts sit in clear tiers — ~1.31% for the departure fields, ~1.38% for the arrival fields, ~1.61% for `arr_delay` / `air_time` / `actual_elapsed_time`, and 98.6% for `cancellation_code`. That layered structure is a strong hint that the missing values are **not random**: they line up with **flight status**. A flight can either operate normally, be **cancelled** (never departs or arrives), or be **diverted** (departs but doesn't complete at the scheduled destination). Below we confirm that every null is explained by one of these cases.

In [8]:
# How many flights were cancelled vs diverted?
flights[["cancelled", "diverted"]].sum()

cancelled    96315
diverted     17499
dtype: int64

In [9]:
# Label each flight by status, then measure the null rate of every column within each group.
# This shows *where* the nulls live: which flight status is responsible for each gap.
status = np.where(flights["cancelled"] == 1, "cancelled",
         np.where(flights["diverted"] == 1, "diverted", "operated"))

null_by_status = flights.isna().groupby(status).mean().round(3).T
null_by_status[null_by_status.sum(axis=1) > 0]   # only columns that actually have nulls

,cancelled,diverted,operated
dep_time,0.962,0.000,0.0
dep_delay,0.965,0.000,0.0
taxi_out,0.994,0.000,0.0
wheels_off,0.994,0.000,0.0
wheels_on,1.000,0.088,0.0
taxi_in,1.000,0.088,0.0
arr_time,1.000,0.088,0.0
arr_delay,1.000,1.000,0.0
cancellation_code,0.000,1.000,1.0
actual_elapsed_time,1.000,1.000,0.0


**Reading the table.** The three columns are the status groups — **cancelled · diverted · operated** — and each value is the *fraction of flights in that group where the column is null* (0 = never missing, 1 = always missing). Every number has a clean physical meaning:

| column | cancelled | diverted | operated |
|---|---|---|---|
| dep_time | 0.962 | 0.000 | 0.0 |
| dep_delay | 0.965 | 0.000 | 0.0 |
| taxi_out | 0.994 | 0.000 | 0.0 |
| wheels_off | 0.994 | 0.000 | 0.0 |
| wheels_on | 1.000 | 0.088 | 0.0 |
| taxi_in | 1.000 | 0.088 | 0.0 |
| arr_time | 1.000 | 0.088 | 0.0 |
| arr_delay | 1.000 | 1.000 | 0.0 |
| cancellation_code | 0.000 | 1.000 | 1.0 |
| actual_elapsed_time | 1.000 | 1.000 | 0.0 |
| air_time | 1.000 | 1.000 | 0.0 |

- **Operated flights** (right column, all `0.0`) are completely clean — zero missing values in every field. The only null is `cancellation_code` (1.0), which is correct: a code exists only for cancelled flights. These are the usable rows.
- **Cancelled flights** (left column) essentially never happened: departure fields are 96–99% null and every arrival field is 100% null, while `cancellation_code` is always present (`0.000` null). The small gradient — `dep_time` 96.2% null vs `taxi_out`/`wheels_off` 99.4% null — shows a thin slice got a departure time logged but were cancelled before actually taxiing or taking off.
- **Diverted flights** (middle column) depart normally (departure fields `0.000` null) and ~91% land somewhere — the diversion airport — so `wheels_on`/`taxi_in`/`arr_time` are only 8.8% null. But `arr_delay`, `air_time`, and `actual_elapsed_time` are 100% null because they are measured against the *scheduled destination*, which was never reached. `cancellation_code` is null since a diverted flight isn't cancelled.

**Takeaway.** The missingness is entirely structural — there are no stray nulls to impute. The target `arr_delay` is missing for **100% of both cancelled and diverted flights** (~1.6% of all rows), so those rows carry no label and cannot train the model. Dropping them with `flights[flights["arr_delay"].notna()]` catches both cases at once; the remaining operated-only data has essentially no missing values, and `cancelled`, `diverted`, and `cancellation_code` become constant/empty on that subset.

### Step four: keep only flights that actually arrived

Because the nulls are structural — they include the **target itself** (`arr_delay`) — we **drop these rows rather than impute them**. You cannot fabricate a label, and filling features for rows that have no label adds nothing to a supervised model.

Rather than filtering on `cancelled` alone, we filter on the target: keep rows where `arr_delay` is present. Since `arr_delay` is null for 100% of *both* cancelled and diverted flights, this single condition removes both (plus any stray label-less row). For this inflight arrival-delay model, cancelled flights carry no usable information — a flight that never operated has no delay to learn from. (Predicting *cancellations* would be a separate model on a separate dataset, and is out of scope here.)

In [10]:
before = len(flights)
flights = flights[flights["arr_delay"].notna()].copy()

dropped = before - len(flights)
print(f"Dropped {dropped:,} rows ({dropped / before:.2%}) with no arr_delay")
print(f"{len(flights):,} operated flights remain")
flights.shape

Dropped 113,814 rows (1.61%) with no arr_delay
6,965,267 operated flights remain


(6965267, 35)

In [11]:
remaining_nulls = flights.isna().sum()
remaining_nulls[remaining_nulls > 0].sort_values(ascending=False)

cancellation_code    6965267
op_carrier_fl_num          1
dtype: int64

With cancelled and diverted flights removed, the operated-flight table is essentially null-free. Two expected leftovers:

- **`cancellation_code` is now 100% null** — no cancelled rows remain — and `cancelled` / `diverted` are now constant. All three are uninformative on this subset and belong in the drop-at-modeling list.
- Any tiny residual (e.g. a stray `crs_elapsed_time` or `op_carrier_fl_num`) is a handful of rows and can simply be dropped or inspected individually.

The dataset is now clean and every remaining row has a valid `arr_delay` target, so we can move on to exploring the delay distribution itself.

### Step five: check for negative values

A negative number is only a problem in context:

- **Legitimate:** `dep_delay` and `arr_delay` go negative when a flight departs or arrives *early*. Early flights are completely normal — we must keep these, since clipping them to zero would erase real signal and bias the target.
- **Impossible:** durations and distances (`taxi_out`, `taxi_in`, `air_time`, `actual_elapsed_time`, `crs_elapsed_time`, `distance`) cannot physically be negative. Any negatives there are data errors worth investigating.

We scan **every column in the dataset** in one pass (numeric columns only — text and dates can't be negative), then read off which negatives are legitimate and which, if any, are errors.

In [12]:
# Scan EVERY column in the dataset for negatives in one pass.
# is_numeric_dtype covers float, int, and pandas' nullable Int64; text/date columns -> 0.
neg_counts = flights.apply(
    lambda c: int((c < 0).sum()) if pd.api.types.is_numeric_dtype(c) else 0
)
neg_counts.sort_values(ascending=False)

arr_delay              4318559
dep_delay              4068725
year                         0
wheels_on                    0
crs_arr_time                 0
arr_time                     0
cancelled                    0
cancellation_code            0
diverted                     0
crs_elapsed_time             0
actual_elapsed_time          0
air_time                     0
distance                     0
carrier_delay                0
weather_delay                0
nas_delay                    0
security_delay               0
taxi_in                      0
wheels_off                   0
month                        0
taxi_out                     0
dep_time                     0
crs_dep_time                 0
dest_state_nm                0
dest_city_name               0
dest                         0
origin_state_nm              0
origin_city_name             0
origin                       0
op_carrier_fl_num            0
op_unique_carrier            0
fl_date                      0
day_of_w

**Result: the data is clean — no impossible negatives.** Every duration, distance, and clock-time column reads `0`. The only negatives are in the two delay columns, and both are entirely legitimate:

- **`arr_delay`: 4,318,559 negatives (~62% of operated flights)** — most flights actually arrive *early*.
- **`dep_delay`: 4,068,725 negatives (~58%)** — most flights depart *early* or on time.

This is expected: airlines pad their schedules, so early arrivals are the norm. We keep these values exactly as they are — a negative delay is real signal, not an error, and clipping it to zero would bias the target.

**Takeaway for modeling:** the target `arr_delay` is centred **below zero** (its median is negative), with a long right tail of genuinely late flights pulling the mean back up toward zero. "Delayed" flights (`arr_delay ≥ 15`) are therefore a minority — important context for how we frame the problem later (a skewed regression target, or an imbalanced classification label).

### Step six: convert `fl_date` to datetime

`fl_date` loaded as an object (plain string). Converting it to a true `datetime64` unlocks time-based analysis — pulling out month / day-of-week, resampling, and sorting chronologically. We pass the explicit `%Y-%m-%d` format so pandas parses directly instead of inferring the layout row by row (much faster on ~7M rows).

In [13]:
flights["fl_date"] = pd.to_datetime(flights["fl_date"], format="%Y-%m-%d")
flights["fl_date"].dtype

dtype('<M8[ns]')

### Step seven: duplicate check

A scheduled flight is uniquely identified by its **date, carrier, flight number, origin, destination, and scheduled departure time**. We check for both fully identical rows and repeated flight keys. If the key count is zero, every row is a distinct flight and our counts and aggregations can be trusted; if not, we'd drop the repeats with `drop_duplicates(subset=flight_key)`.

In [14]:
flight_key = ["fl_date", "op_unique_carrier", "op_carrier_fl_num",
              "origin", "dest", "crs_dep_time"]

dup_rows = flights.duplicated().sum()                    # fully identical rows
dup_keys = flights.duplicated(subset=flight_key).sum()   # same scheduled flight
print(f"Fully duplicated rows: {dup_rows:,}")
print(f"Duplicate flight keys:  {dup_keys:,}")

Fully duplicated rows: 0
Duplicate flight keys:  0


### Step eight: optimize memory (categoricals)

The text columns repeat the same handful of values millions of times — only a few dozen carriers and a few hundred airports/states across ~7M rows. Casting them to pandas `category` replaces the repeated strings with compact integer codes plus a small lookup table, which cuts memory sharply and makes `groupby` / `value_counts` faster throughout the EDA. We convert every low-cardinality string column and report the reduction.

In [15]:
before_mb = flights.memory_usage(deep=True).sum() / 1024**2

cat_cols = ["op_unique_carrier", "origin", "origin_city_name", "origin_state_nm",
            "dest", "dest_city_name", "dest_state_nm"]
flights[cat_cols] = flights[cat_cols].astype("category")

after_mb = flights.memory_usage(deep=True).sum() / 1024**2
print(f"Memory: {before_mb:,.0f} MB -> {after_mb:,.0f} MB "
      f"({1 - after_mb / before_mb:.0%} smaller)")
flights.dtypes

Memory: 4,314 MB -> 1,774 MB (59% smaller)


year                            int64
month                           int64
day_of_month                    int64
day_of_week                     int64
fl_date                datetime64[ns]
op_unique_carrier            category
op_carrier_fl_num             float64
origin                       category
origin_city_name             category
origin_state_nm              category
dest                         category
dest_city_name               category
dest_state_nm                category
crs_dep_time                    int64
dep_time                      float64
dep_delay                     float64
taxi_out                      float64
wheels_off                    float64
wheels_on                     float64
taxi_in                       float64
crs_arr_time                    int64
arr_time                      float64
arr_delay                     float64
cancelled                       int64
cancellation_code              object
diverted                        int64
crs_elapsed_